# Named Entity Recognition — LSTM vs BiLSTM vs BiLSTM+CRF vs Fine-tuned Transformer (CoNLL-2003)

In [1]:
!pip install -q -U datasets evaluate seqeval pytorch-crf gradio transformers accelerate gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.2 MB/s eta 0:00:00


In [2]:
import os, re, random, pickle, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

from datasets import load_dataset
import evaluate
from seqeval.metrics import classification_report as seq_classification_report
from seqeval.metrics import f1_score as seq_f1_score, precision_score as seq_precision_score, recall_score as seq_recall_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

os.makedirs("models", exist_ok=True)

cuda


## 1. Load CoNLL-2003 dataset

In [6]:
raw_datasets = load_dataset("lhoestq/conll2003")
print(raw_datasets)

label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}
NUM_LABELS = len(label_list)
print(label_list)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [7]:
sample = raw_datasets["train"][0]
for tok, tag in zip(sample["tokens"], sample["ner_tags"]):
    print(tok, id2label[tag])

EU B-ORG
rejects O
German B-MISC
call O
to O
boycott O
British B-MISC
lamb O
. O


## 2. Build vocabularies (word-level + char-level for OOV handling)

In [8]:
from collections import Counter

word_counter = Counter()
char_set = set()
for ex in raw_datasets["train"]:
    for tok in ex["tokens"]:
        word_counter[tok.lower()] += 1
        char_set.update(list(tok))

PAD, UNK = "<PAD>", "<UNK>"
word2idx = {PAD: 0, UNK: 1}
for w, c in word_counter.most_common():
    word2idx[w] = len(word2idx)

char2idx = {PAD: 0, UNK: 1}
for ch in sorted(char_set):
    char2idx[ch] = len(char2idx)

VOCAB_SIZE = len(word2idx)
CHAR_VOCAB_SIZE = len(char2idx)
MAX_WORD_LEN = 20
print(VOCAB_SIZE, CHAR_VOCAB_SIZE)

with open("models/word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)
with open("models/char2idx.pkl", "wb") as f:
    pickle.dump(char2idx, f)
with open("models/label_list.pkl", "wb") as f:
    pickle.dump(label_list, f)

21011 86


## 3. Pre-trained GloVe embeddings

In [9]:
import gensim.downloader as gensim_api

glove = gensim_api.load("glove-wiki-gigaword-100")
EMBED_DIM = 100

embedding_matrix = np.random.normal(scale=0.1, size=(VOCAB_SIZE, EMBED_DIM)).astype(np.float32)
embedding_matrix[0] = np.zeros(EMBED_DIM)
hits = 0
for w, idx in word2idx.items():
    if w in glove.key_to_index:
        embedding_matrix[idx] = glove[w]
        hits += 1
print(f"GloVe coverage: {hits}/{VOCAB_SIZE}")

[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe coverage: 18415/21011


## 4. PyTorch Dataset + collate function

In [10]:
MAX_LEN = 128

def encode_example(tokens, tags):
    word_ids = [word2idx.get(t.lower(), word2idx[UNK]) for t in tokens][:MAX_LEN]
    char_ids = []
    for t in tokens[:MAX_LEN]:
        cids = [char2idx.get(c, char2idx[UNK]) for c in t[:MAX_WORD_LEN]]
        cids = cids + [char2idx[PAD]] * (MAX_WORD_LEN - len(cids))
        char_ids.append(cids)
    label_ids = tags[:MAX_LEN]
    return word_ids, char_ids, label_ids

class NERDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        ex = self.data[i]
        word_ids, char_ids, label_ids = encode_example(ex["tokens"], ex["ner_tags"])
        return {"word_ids": word_ids, "char_ids": char_ids, "label_ids": label_ids}

def collate_fn(batch):
    max_len = max(len(b["word_ids"]) for b in batch)
    word_batch, char_batch, label_batch, mask_batch = [], [], [], []
    for b in batch:
        pad_len = max_len - len(b["word_ids"])
        word_batch.append(b["word_ids"] + [0] * pad_len)
        char_batch.append(b["char_ids"] + [[0] * MAX_WORD_LEN] * pad_len)
        label_batch.append(b["label_ids"] + [-100] * pad_len)
        mask_batch.append([1] * len(b["word_ids"]) + [0] * pad_len)
    return (
        torch.tensor(word_batch, dtype=torch.long),
        torch.tensor(char_batch, dtype=torch.long),
        torch.tensor(label_batch, dtype=torch.long),
        torch.tensor(mask_batch, dtype=torch.bool),
    )

train_ds = NERDataset(raw_datasets["train"])
val_ds = NERDataset(raw_datasets["validation"])
test_ds = NERDataset(raw_datasets["test"])

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 5. Char-level CNN encoder (handles OOV words)

In [11]:
class CharCNN(nn.Module):
    def __init__(self, char_vocab_size, char_embed_dim=25, out_channels=30, kernel_size=3):
        super().__init__()
        self.char_embed = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        self.conv = nn.Conv1d(char_embed_dim, out_channels, kernel_size, padding=kernel_size // 2)
        self.out_dim = out_channels

    def forward(self, char_ids):
        B, T, C = char_ids.shape
        x = self.char_embed(char_ids.view(B * T, C))
        x = x.transpose(1, 2)
        x = torch.relu(self.conv(x))
        x = torch.max(x, dim=2).values
        return x.view(B, T, self.out_dim)

## 6. Model architectures

In [12]:
class LSTMTagger(nn.Module):
    def __init__(self, embedding_matrix, char_vocab_size, num_labels, hidden_dim=128, bidirectional=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.word_embed = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=False, padding_idx=0)
        self.char_cnn = CharCNN(char_vocab_size)
        rnn_in = embed_dim + self.char_cnn.out_dim
        self.lstm = nn.LSTM(rnn_in, hidden_dim, batch_first=True, bidirectional=bidirectional)
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(out_dim, num_labels)

    def forward(self, word_ids, char_ids):
        we = self.word_embed(word_ids)
        ce = self.char_cnn(char_ids)
        x = torch.cat([we, ce], dim=-1)
        x, _ = self.lstm(x)
        x = self.dropout(x)
        return self.fc(x)


class BiLSTM_CRF(nn.Module):
    def __init__(self, embedding_matrix, char_vocab_size, num_labels, hidden_dim=128):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.word_embed = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=False, padding_idx=0)
        self.char_cnn = CharCNN(char_vocab_size)
        rnn_in = embed_dim + self.char_cnn.out_dim
        self.lstm = nn.LSTM(rnn_in, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def emissions(self, word_ids, char_ids):
        we = self.word_embed(word_ids)
        ce = self.char_cnn(char_ids)
        x = torch.cat([we, ce], dim=-1)
        x, _ = self.lstm(x)
        x = self.dropout(x)
        return self.fc(x)

    def loss(self, word_ids, char_ids, labels, mask):
        emissions = self.emissions(word_ids, char_ids)
        safe_labels = labels.clone()
        safe_labels[~mask] = 0
        return -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

    def decode(self, word_ids, char_ids, mask):
        emissions = self.emissions(word_ids, char_ids)
        return self.crf.decode(emissions, mask=mask)

## 7. Generic training + seqeval evaluation

In [13]:
results = {}

def ids_to_labels(pred_ids, true_ids):
    pred_labels, true_labels = [], []
    for p_seq, t_seq in zip(pred_ids, true_ids):
        p_lbls, t_lbls = [], []
        for p, t in zip(p_seq, t_seq):
            if t == -100:
                continue
            p_lbls.append(id2label[p])
            t_lbls.append(id2label[t])
        pred_labels.append(p_lbls)
        true_labels.append(t_lbls)
    return pred_labels, true_labels

def evaluate_loader(model, loader, is_crf=False):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for word_ids, char_ids, labels, mask in loader:
            word_ids, char_ids, labels, mask = word_ids.to(device), char_ids.to(device), labels.to(device), mask.to(device)
            if is_crf:
                pred_seqs = model.decode(word_ids, char_ids, mask)
                pred_ids = [seq + [0] * (labels.shape[1] - len(seq)) for seq in pred_seqs]
            else:
                logits = model(word_ids, char_ids)
                pred_ids = torch.argmax(logits, dim=-1).cpu().tolist()
            all_preds.extend(pred_ids)
            all_trues.extend(labels.cpu().tolist())
    pred_labels, true_labels = ids_to_labels(all_preds, all_trues)
    return pred_labels, true_labels

def train_and_eval(name, model, epochs=6, lr=1e-3, is_crf=False):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    ce_loss = nn.CrossEntropyLoss(ignore_index=-100)
    best_f1, best_state = -1, None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for word_ids, char_ids, labels, mask in train_loader:
            word_ids, char_ids, labels, mask = word_ids.to(device), char_ids.to(device), labels.to(device), mask.to(device)
            optimizer.zero_grad()
            if is_crf:
                loss = model.loss(word_ids, char_ids, labels, mask)
            else:
                logits = model(word_ids, char_ids)
                loss = ce_loss(logits.view(-1, NUM_LABELS), labels.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        pred_labels, true_labels = evaluate_loader(model, val_loader, is_crf=is_crf)
        val_f1 = seq_f1_score(true_labels, pred_labels)
        print(f"[{name}] epoch {epoch+1}/{epochs} loss={total_loss/len(train_loader):.4f} val_f1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(best_state, f"models/best_{name}.pt")

    pred_labels, true_labels = evaluate_loader(model, test_loader, is_crf=is_crf)
    print(f"\n=== {name.upper()} — test set ===")
    print(seq_classification_report(true_labels, pred_labels, digits=4))

    results[name] = {
        "precision": seq_precision_score(true_labels, pred_labels),
        "recall": seq_recall_score(true_labels, pred_labels),
        "f1": seq_f1_score(true_labels, pred_labels),
    }
    return model

## 8. Train LSTM

In [14]:
lstm_model = LSTMTagger(embedding_matrix, CHAR_VOCAB_SIZE, NUM_LABELS, bidirectional=False)
lstm_model = train_and_eval("lstm", lstm_model, is_crf=False)

[lstm] epoch 1/6 loss=0.3627 val_f1=0.7666
[lstm] epoch 2/6 loss=0.0952 val_f1=0.8100
[lstm] epoch 3/6 loss=0.0633 val_f1=0.8156
[lstm] epoch 4/6 loss=0.0473 val_f1=0.8154
[lstm] epoch 5/6 loss=0.0370 val_f1=0.8317
[lstm] epoch 6/6 loss=0.0295 val_f1=0.8210

=== LSTM — test set ===
              precision    recall  f1-score   support

         LOC     0.7513    0.8747    0.8083      1668
        MISC     0.6121    0.6766    0.6428       702
         ORG     0.6754    0.6201    0.6466      1661
         PER     0.7885    0.8924    0.8372      1617

   micro avg     0.7257    0.7803    0.7520      5648
   macro avg     0.7068    0.7660    0.7337      5648
weighted avg     0.7223    0.7803    0.7485      5648



## 9. Train BiLSTM

In [15]:
bilstm_model = LSTMTagger(embedding_matrix, CHAR_VOCAB_SIZE, NUM_LABELS, bidirectional=True)
bilstm_model = train_and_eval("bilstm", bilstm_model, is_crf=False)

[bilstm] epoch 1/6 loss=0.2967 val_f1=0.8208
[bilstm] epoch 2/6 loss=0.0728 val_f1=0.8575
[bilstm] epoch 3/6 loss=0.0396 val_f1=0.8728
[bilstm] epoch 4/6 loss=0.0242 val_f1=0.8840
[bilstm] epoch 5/6 loss=0.0155 val_f1=0.8884
[bilstm] epoch 6/6 loss=0.0105 val_f1=0.8766

=== BILSTM — test set ===
              precision    recall  f1-score   support

         LOC     0.8690    0.8987    0.8836      1668
        MISC     0.6591    0.7493    0.7013       702
         ORG     0.7742    0.7556    0.7648      1661
         PER     0.8624    0.8763    0.8693      1617

   micro avg     0.8116    0.8316    0.8215      5648
   macro avg     0.7912    0.8200    0.8048      5648
weighted avg     0.8132    0.8316    0.8219      5648



## 10. Train BiLSTM + CRF

In [16]:
bilstm_crf_model = BiLSTM_CRF(embedding_matrix, CHAR_VOCAB_SIZE, NUM_LABELS)
bilstm_crf_model = train_and_eval("bilstm_crf", bilstm_crf_model, is_crf=True)

[bilstm_crf] epoch 1/6 loss=4.0207 val_f1=0.8333
[bilstm_crf] epoch 2/6 loss=0.9578 val_f1=0.8776
[bilstm_crf] epoch 3/6 loss=0.5147 val_f1=0.8950
[bilstm_crf] epoch 4/6 loss=0.3131 val_f1=0.8923
[bilstm_crf] epoch 5/6 loss=0.1889 val_f1=0.9023
[bilstm_crf] epoch 6/6 loss=0.1312 val_f1=0.8953

=== BILSTM_CRF — test set ===
              precision    recall  f1-score   support

         LOC     0.8629    0.8981    0.8801      1668
        MISC     0.7813    0.7379    0.7590       702
         ORG     0.8438    0.7706    0.8055      1661
         PER     0.8888    0.8992    0.8939      1617

   micro avg     0.8555    0.8410    0.8482      5648
   macro avg     0.8442    0.8264    0.8346      5648
weighted avg     0.8545    0.8410    0.8471      5648



## 11. Fine-tune HuggingFace Transformer (token classification)

In [17]:
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)

MODEL_NAME = "distilbert-base-cased"
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(examples):
    tokenized = hf_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=MAX_LEN)
    all_labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word_id:
                label_ids.append(tags[wid])
            else:
                label_ids.append(-100)
            prev_word_id = wid
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_datasets = raw_datasets.map(tokenize_and_align, batched=True, remove_columns=raw_datasets["train"].column_names)
data_collator = DataCollatorForTokenClassification(tokenizer=hf_tokenizer)

hf_model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id)

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        p_lbls = [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        t_lbls = [id2label[l] for p, l in zip(pred_seq, label_seq) if l != -100]
        true_preds.append(p_lbls)
        true_labels.append(t_lbls)
    scores = seqeval_metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": scores["overall_precision"],
        "recall": scores["overall_recall"],
        "f1": scores["overall_f1"],
        "accuracy": scores["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir="models/transformer_ckpts",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=hf_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=hf_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3512, training_loss=0.043207219150553804, metrics={'train_runtime': 179.5408, 'train_samples_per_second': 312.82, 'train_steps_per_second': 19.561, 'total_flos': 700240717592892.0, 'train_loss': 0.043207219150553804, 'epoch': 4.0})

In [18]:
best_transformer_dir = "models/best_transformer"
trainer.save_model(best_transformer_dir)
hf_tokenizer.save_pretrained(best_transformer_dir)

test_metrics = trainer.evaluate(tokenized_datasets["test"])
print(test_metrics)

test_preds = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(test_preds.predictions, axis=-1)
labels = test_preds.label_ids
true_preds, true_labels = [], []
for pred_seq, label_seq in zip(preds, labels):
    p_lbls = [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
    t_lbls = [id2label[l] for p, l in zip(pred_seq, label_seq) if l != -100]
    true_preds.append(p_lbls)
    true_labels.append(t_lbls)

print(seq_classification_report(true_labels, true_preds, digits=4))
results["transformer"] = {
    "precision": seq_precision_score(true_labels, true_preds),
    "recall": seq_recall_score(true_labels, true_preds),
    "f1": seq_f1_score(true_labels, true_preds),
}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.008009,0.120509,4,0.889643,0.905563,0.897533,0.979296


{'eval_loss': 0.1205093264579773, 'eval_precision': 0.8896431679721497, 'eval_recall': 0.9055634301913537, 'eval_f1': 0.8975327069979805, 'eval_accuracy': 0.9792963784820217}


              precision    recall  f1-score   support

         LOC     0.9345    0.9166    0.9255      1666
        MISC     0.7543    0.8134    0.7827       702
         ORG     0.8545    0.8910    0.8724      1661
         PER     0.9451    0.9492    0.9472      1615

   micro avg     0.8896    0.9056    0.8975      5644
   macro avg     0.8721    0.8926    0.8819      5644
weighted avg     0.8916    0.9056    0.8983      5644



## 12. Compare architectures

In [19]:
import pandas as pd

results_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print(results_df)

with open("models/results_summary.json", "w") as f:
    json.dump(results, f, indent=2)

best_model_name = results_df.index[0]
print("Best model:", best_model_name)
# CRF typically raises boundary-sensitive metrics (fewer split B-/I- errors)
# because it models valid tag transitions jointly instead of per-token independently.

             precision    recall        f1
transformer   0.889643  0.905563  0.897533
bilstm_crf    0.855548  0.841006  0.848214
bilstm        0.811647  0.831622  0.821513
lstm          0.725671  0.780276  0.751984
Best model: transformer


## 13. Inference helper (word-level tokenization + span reconstruction)

In [20]:
def simple_tokenize_with_offsets(text):
    tokens, offsets = [], []
    for m in re.finditer(r"\S+", text):
        tokens.append(m.group())
        offsets.append((m.start(), m.end()))
    return tokens, offsets

def predict_spans(text, model_name=None):
    model_name = model_name or best_model_name

    if model_name == "transformer":
        model = AutoModelForTokenClassification.from_pretrained(best_transformer_dir).to(device).eval()
        tok = AutoTokenizer.from_pretrained(best_transformer_dir)
        enc = tok(text, return_tensors="pt", return_offsets_mapping=True, truncation=True, max_length=MAX_LEN)
        offset_mapping = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()
        spans = []
        for (start, end), pid in zip(offset_mapping, pred_ids):
            if start == end:
                continue
            spans.append((start, end, id2label[pid]))
        return spans

    tokens, offsets = simple_tokenize_with_offsets(text)
    word_ids = [word2idx.get(t.lower(), word2idx[UNK]) for t in tokens]
    char_ids = []
    for t in tokens:
        cids = [char2idx.get(c, char2idx[UNK]) for c in t[:MAX_WORD_LEN]]
        cids = cids + [char2idx[PAD]] * (MAX_WORD_LEN - len(cids))
        char_ids.append(cids)

    word_tensor = torch.tensor([word_ids], dtype=torch.long).to(device)
    char_tensor = torch.tensor([char_ids], dtype=torch.long).to(device)
    mask_tensor = torch.tensor([[1] * len(word_ids)], dtype=torch.bool).to(device)

    if model_name == "lstm":
        model = lstm_model.eval()
        with torch.no_grad():
            logits = model(word_tensor, char_tensor)
        pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()
    elif model_name == "bilstm":
        model = bilstm_model.eval()
        with torch.no_grad():
            logits = model(word_tensor, char_tensor)
        pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()
    else:
        model = bilstm_crf_model.eval()
        with torch.no_grad():
            pred_ids = model.decode(word_tensor, char_tensor, mask_tensor)[0]

    return [(offsets[i][0], offsets[i][1], id2label[pid]) for i, pid in enumerate(pred_ids)]

## 14. Gradio app — real-time color-coded entity highlighting

In [21]:
import gradio as gr

ENTITY_COLORS = {"PER": "#ff9999", "ORG": "#99ccff", "LOC": "#99ff99", "MISC": "#ffe599"}

def gradio_predict(text, model_choice):
    spans = predict_spans(text, model_name=None if model_choice == "Best model" else model_choice)
    entities = []
    for start, end, tag in spans:
        if tag == "O":
            continue
        entities.append({"entity": tag[2:], "start": start, "end": end})

    merged = []
    for ent in entities:
        if merged and merged[-1]["entity"] == ent["entity"] and ent["start"] - merged[-1]["end"] <= 1:
            merged[-1]["end"] = ent["end"]
        else:
            merged.append(dict(ent))

    highlighted = []
    cursor = 0
    for ent in merged:
        if ent["start"] > cursor:
            highlighted.append((text[cursor:ent["start"]], None))
        highlighted.append((text[ent["start"]:ent["end"]], ent["entity"]))
        cursor = ent["end"]
    if cursor < len(text):
        highlighted.append((text[cursor:], None))
    return highlighted

model_options = ["Best model", "lstm", "bilstm", "bilstm_crf", "transformer"]

demo = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Textbox(lines=4, label="Enter text", placeholder="e.g. Elon Musk visited Berlin to meet officials from the German government."),
        gr.Dropdown(model_options, value="Best model", label="Model"),
    ],
    outputs=gr.HighlightedText(label="Detected entities", color_map=ENTITY_COLORS),
    title="Named Entity Recognition",
    description="LSTM / BiLSTM / BiLSTM+CRF / Fine-tuned Transformer — CoNLL-2003",
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://619ba1c0f7b701b328.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
